In [1]:
from gensim.models import Word2Vec
import pandas as pd
import numpy as np
from w2v_train import review_to_wordlist
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

D:\Codding\Education\NLP\Bag of Words Meets Bags of Popcorn\w2v_train.py:21: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 21 of the file D:\Codding\Education\NLP\Bag of Words Meets Bags of Popcorn\w2v_train.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  soup = BeautifulSoup(raw_text)
D:\Codding\Education\NLP\Bag of Words Meets Bags of Popcorn\w2v_train.py:21: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'r

Инициализация и сборка словаря Word2Vec...


2026-07-06 17:49:52,061 : INFO : PROGRESS: at sentence #70000, processed 1560306 words, keeping 43311 word types
2026-07-06 17:49:52,091 : INFO : PROGRESS: at sentence #80000, processed 1779515 words, keeping 45707 word types
2026-07-06 17:49:52,122 : INFO : PROGRESS: at sentence #90000, processed 2003713 words, keeping 48121 word types
2026-07-06 17:49:52,152 : INFO : PROGRESS: at sentence #100000, processed 2225464 words, keeping 50190 word types
2026-07-06 17:49:52,179 : INFO : PROGRESS: at sentence #110000, processed 2444322 words, keeping 52058 word types
2026-07-06 17:49:52,212 : INFO : PROGRESS: at sentence #120000, processed 2666487 words, keeping 54098 word types
2026-07-06 17:49:52,240 : INFO : PROGRESS: at sentence #130000, processed 2892314 words, keeping 55837 word types
2026-07-06 17:49:52,261 : INFO : PROGRESS: at sentence #140000, processed 3104795 words, keeping 57324 word types
2026-07-06 17:49:52,287 : INFO : PROGRESS: at sentence #150000, processed 3330431 words, ke

Обучение Word2Vec (это займет пару минут)...


2026-07-06 17:49:54,563 : INFO : EPOCH 0 - PROGRESS: at 20.20% examples, 1692104 words/s, in_qsize 7, out_qsize 0
2026-07-06 17:49:55,567 : INFO : EPOCH 0 - PROGRESS: at 39.32% examples, 1646416 words/s, in_qsize 7, out_qsize 0
2026-07-06 17:49:56,568 : INFO : EPOCH 0 - PROGRESS: at 59.78% examples, 1669278 words/s, in_qsize 7, out_qsize 0
2026-07-06 17:49:57,571 : INFO : EPOCH 0 - PROGRESS: at 81.16% examples, 1698573 words/s, in_qsize 7, out_qsize 0
2026-07-06 17:49:58,490 : INFO : EPOCH 0: training on 11841450 raw words (8391336 effective words) took 4.9s, 1702245 effective words/s
2026-07-06 17:49:59,496 : INFO : EPOCH 1 - PROGRESS: at 20.12% examples, 1686380 words/s, in_qsize 7, out_qsize 0
2026-07-06 17:50:00,497 : INFO : EPOCH 1 - PROGRESS: at 41.15% examples, 1726167 words/s, in_qsize 7, out_qsize 0
2026-07-06 17:50:01,511 : INFO : EPOCH 1 - PROGRESS: at 62.33% examples, 1734099 words/s, in_qsize 7, out_qsize 0
2026-07-06 17:50:02,513 : INFO : EPOCH 1 - PROGRESS: at 83.11% exa

Word2Vec успешно обучен и сохранен!


In [9]:
train = pd.read_csv("data/labeledTrainData.tsv", header=0,
                    delimiter="\t", quoting=3)

test = pd.read_csv("data/testData.tsv", header=0,
                   delimiter="\t", quoting=3)

In [3]:
num_features = 300    # Word vector dimensionality
model_w2v = Word2Vec.load("models/300features_40minwords_10context_w2v")

2026-07-06 17:50:18,686 : INFO : loading Word2Vec object from models/300features_40minwords_10context_w2v
2026-07-06 17:50:18,695 : INFO : loading wv recursively from models/300features_40minwords_10context_w2v.wv.* with mmap=None
2026-07-06 17:50:18,695 : INFO : setting ignored attribute cum_table to None
2026-07-06 17:50:18,758 : INFO : Word2Vec lifecycle event {'fname': 'models/300features_40minwords_10context_w2v', 'datetime': '2026-07-06T17:50:18.758322', 'gensim': '4.4.0', 'python': '3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'loaded'}


In [4]:
def make_feature_vec(words, model, num_features):
    feature_vec = np.zeros(num_features,dtype="float32")
    nwords = 0
    index2key_set = set(model.wv.index_to_key)

    for word in words:
        if word in index2key_set:
            nwords = nwords + 1
            feature_vec = np.add(feature_vec, model.wv[word])

    feature_vec = np.divide(feature_vec, nwords)

    return feature_vec

In [5]:
def get_avg_feature_vecs(reviews, model, num_features):
    counter = 0
    review_feature_vecs = np.zeros((len(reviews), num_features), dtype="float32")

    for review in reviews:
       if counter % 1000 == 0:
           print("Review %d of %d" % (counter, len(reviews)))

       review_feature_vecs[counter] = make_feature_vec(review, model, num_features)
       counter = counter + 1

    return review_feature_vecs

In [6]:
clean_train_reviews = []
for review in train["review"]:
    clean_train_reviews.append(review_to_wordlist(review, remove_stopwords=True))

train_data_vecs = get_avg_feature_vecs(clean_train_reviews, model_w2v, num_features)

print("Creating average feature vecs for test reviews")
clean_test_reviews = []
for review in test["review"]:
    clean_test_reviews.append(review_to_wordlist(review, remove_stopwords=True))

test_data_vecs = get_avg_feature_vecs(clean_test_reviews, model_w2v, num_features)

D:\Codding\Education\NLP\Bag of Words Meets Bags of Popcorn\w2v_train.py:21: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 21 of the file D:\Codding\Education\NLP\Bag of Words Meets Bags of Popcorn\w2v_train.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  soup = BeautifulSoup(raw_text)


Review 0 of 25000
Review 1000 of 25000
Review 2000 of 25000
Review 3000 of 25000
Review 4000 of 25000
Review 5000 of 25000
Review 6000 of 25000
Review 7000 of 25000
Review 8000 of 25000
Review 9000 of 25000
Review 10000 of 25000
Review 11000 of 25000
Review 12000 of 25000
Review 13000 of 25000
Review 14000 of 25000
Review 15000 of 25000
Review 16000 of 25000
Review 17000 of 25000
Review 18000 of 25000
Review 19000 of 25000
Review 20000 of 25000
Review 21000 of 25000
Review 22000 of 25000
Review 23000 of 25000
Review 24000 of 25000
Creating average feature vecs for test reviews
Review 0 of 25000
Review 1000 of 25000
Review 2000 of 25000
Review 3000 of 25000
Review 4000 of 25000
Review 5000 of 25000
Review 6000 of 25000
Review 7000 of 25000
Review 8000 of 25000
Review 9000 of 25000
Review 10000 of 25000
Review 11000 of 25000
Review 12000 of 25000
Review 13000 of 25000
Review 14000 of 25000
Review 15000 of 25000
Review 16000 of 25000
Review 17000 of 25000
Review 18000 of 25000
Review 1900

In [7]:
forest = RandomForestClassifier(n_estimators = 100)

print("Fitting a random forest to labeled training data...")
forest = forest.fit(train_data_vecs, train["sentiment"])

result = forest.predict(test_data_vecs)

output = pd.DataFrame( data={"id":test["id"], "sentiment":result} )
output.to_csv("results/Word2Vec_AverageVectors.csv", index=False, quoting=3)

Fitting a random forest to labeled training data...


# _Metrics_

In [8]:
cv_model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

accuracy_scores = cross_val_score(cv_model, train_data_vecs, train['sentiment'], cv=5, scoring='accuracy')
print(f"Кросс-валидация Accuracy: {accuracy_scores.mean() * 100:.2f}% (разброс: +/- {accuracy_scores.std() * 100:.2f}%)")

roc_auc_scores = cross_val_score(cv_model, train_data_vecs, train['sentiment'], cv=5, scoring='roc_auc')
print(f"Кросс-валидация ROC AUC:  {roc_auc_scores.mean() * 100:.2f}%")

Кросс-валидация Accuracy: 83.36% (разброс: +/- 0.36%)
Кросс-валидация ROC AUC:  90.92%
